# Load Packages

In [1]:
import pandas as pd
import numpy as np
import pickle as pickle
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import matthews_corrcoef, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

# Import Data

In [2]:
train_set = pd.read_csv("../data/clean/twitter_training_clean.csv", index_col=0)
test_set = pd.read_csv("../data/clean/twitter_test_clean.csv", index_col=0)
val_set = pd.read_csv("../data/clean/twitter_validation_clean.csv", index_col=0)

X_train = train_set.drop(labels="Sentiment", axis=1)
y_train = train_set["Sentiment"]

X_test = test_set.drop(labels="Sentiment", axis=1)
y_test = test_set["Sentiment"]

X_val = val_set.drop(labels="Sentiment", axis=1)
y_val = val_set["Sentiment"]

## Encoding

In [3]:
encoder = OrdinalEncoder()
encoder = encoder.fit(X_train)
X_train = encoder.transform(X_train)

encoder = encoder.fit(X_val)
X_val = encoder.transform(X_val)

encoder = encoder.fit(X_test)
X_test = encoder.transform(X_test)

## Scaling

In [4]:
scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train)
X_val = scaler.fit_transform(X_val)

# Model

In [5]:
knn = KNeighborsClassifier()
matthews_scorer = make_scorer(matthews_corrcoef)

## Hyperparameter Search

In [6]:
param_grid = {
    "n_neighbors": [1, 2, 3, 5, 7, 9, 11, 13],
    "algorithm": ["ball_tree", "kd_tree"],
    "weights": ["uniform", "distance"]
}

grid_search = GridSearchCV(knn, param_grid, scoring=matthews_scorer, verbose=3)
grid_search.fit(X_train, y_train)

grid_search.cv_results_

Fitting 5 folds for each of 32 candidates, totalling 160 fits
[CV 1/5] END algorithm=ball_tree, n_neighbors=1, weights=uniform;, score=0.761 total time=   0.6s
[CV 2/5] END algorithm=ball_tree, n_neighbors=1, weights=uniform;, score=0.768 total time=   0.6s
[CV 3/5] END algorithm=ball_tree, n_neighbors=1, weights=uniform;, score=0.765 total time=   0.5s
[CV 4/5] END algorithm=ball_tree, n_neighbors=1, weights=uniform;, score=0.766 total time=   0.5s
[CV 5/5] END algorithm=ball_tree, n_neighbors=1, weights=uniform;, score=0.772 total time=   0.5s
[CV 1/5] END algorithm=ball_tree, n_neighbors=1, weights=distance;, score=0.761 total time=   0.5s
[CV 2/5] END algorithm=ball_tree, n_neighbors=1, weights=distance;, score=0.768 total time=   0.5s
[CV 3/5] END algorithm=ball_tree, n_neighbors=1, weights=distance;, score=0.765 total time=   0.5s
[CV 4/5] END algorithm=ball_tree, n_neighbors=1, weights=distance;, score=0.766 total time=   0.4s
[CV 5/5] END algorithm=ball_tree, n_neighbors=1, wei

{'mean_fit_time': array([0.19355159, 0.18488107, 0.17733555, 0.1485117 , 0.16615705,
        0.18973436, 0.1581347 , 0.14218616, 0.16722412, 0.17007461,
        0.16498613, 0.14434915, 0.13713589, 0.15219412, 0.15307727,
        0.16797791, 0.16496191, 0.17686582, 0.16563773, 0.18001518,
        0.17105436, 0.16509967, 0.22200394, 0.17124305, 0.16486306,
        0.19881101, 0.19862761, 0.19495201, 0.19130502, 0.17180409,
        0.26363454, 0.2842629 ]),
 'std_fit_time': array([0.01136174, 0.03913154, 0.0080291 , 0.02129436, 0.03064088,
        0.01304582, 0.01149889, 0.01974076, 0.02325048, 0.02305777,
        0.03179925, 0.01926953, 0.01859934, 0.02523778, 0.02453822,
        0.00984259, 0.01253491, 0.01850926, 0.00886667, 0.02024655,
        0.01276583, 0.00812072, 0.03489225, 0.01276081, 0.01607769,
        0.01359189, 0.01623285, 0.01930824, 0.02178972, 0.01032306,
        0.05192368, 0.02056054]),
 'mean_score_time': array([0.44892502, 0.43555551, 0.38348799, 0.32431293, 0.390716

In [7]:
best_model = grid_search.best_estimator_

## Training

In [12]:
train_score = best_model.score(X_train, y_train)
print("Training Score: ", train_score)

Training Score:  1.0


## Validation

In [13]:
val_score = best_model.score(X_val, y_val)
print("Validation Score: ", val_score)

Validation Score:  0.6300675675675675


## Test

In [14]:
test_score = best_model.score(X_test, y_test)
print("Test Score: ", test_score)

Test Score:  0.313
